In [21]:
import pyarrow.parquet as pq
import pandas as pd

PATH_RAW = '../Data Layer/raw/fipex-prices-latest-merged.parquet'
PATH_SILVER = '../Data Layer/silver/fipe_silver.parquet'

FILTROS = [
    ('ano_referencia', '>=', 2016),   # 2021–2026 = últimos 5 anos
    ('valor_centavos', '>', 0),
]

COLUNAS = [
    'nome_marca', 'nome_modelo', 'tipo_veiculo',
    'nome_combustivel', 'ano_referencia', 'valor_centavos','ano_modelo'
]

df = pq.read_table(PATH_RAW, columns=COLUNAS, filters=FILTROS).to_pandas()

df = df[df['ano_modelo'].notna()].copy()

# Transformações Silver
df['valor_reais'] = df['valor_centavos'] / 100
df = df.drop(columns=['valor_centavos'])  # tipo já fixo = carro
df['nome_marca']       = df['nome_marca'].str.strip().str.upper()
df['nome_modelo']      = df['nome_modelo'].str.strip().str.title()
df['nome_combustivel'] = df['nome_combustivel'].str.strip().str.title()
df['nome_marca']       = df['nome_marca'].astype('category')
df['tipo_veiculo'] = df['tipo_veiculo'].astype('category')
df['nome_combustivel'] = df['nome_combustivel'].astype('category')
df['ano_referencia']   = df['ano_referencia'].astype('int16')
df['idade_veiculo'] = df['ano_referencia'] - df['ano_modelo']

df.to_parquet(PATH_SILVER, index=False, compression='zstd')

print(f"Shape final: {df.shape}")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Shape final: (5018080, 8)
Memória: 585.0 MB


In [22]:
df.head()

,nome_marca,nome_modelo,tipo_veiculo,nome_combustivel,ano_referencia,ano_modelo,valor_reais,idade_veiculo
0,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3231.0,26.0
1,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3181.0,26.0
2,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3189.0,26.0
3,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3197.0,26.0
4,ADLY,Atv 100,moto,Gasolina,2025,2000.0,3204.0,25.0


In [23]:
# nome_modelo tem muitos valores únicos, mas category ainda ajuda
df['nome_modelo'] = df['nome_modelo'].astype('category')
print(f"Memória após: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Memória após: 201.9 MB


In [24]:
distribuicao_marcas = (df['nome_marca'].value_counts(normalize=True) * 100).round(2).reset_index()
distribuicao_marcas.columns = ['Marca', 'Porcentagem (%)']

contagem_modelos = df.groupby('nome_marca')['nome_modelo'].nunique().reset_index()
contagem_modelos.columns = ['Marca', 'QTD Modelos']

distribuicao_marcas = pd.merge(distribuicao_marcas,contagem_modelos, on='Marca')
distribuicao_marcas.head(10)

C:\Users\nicol\AppData\Local\Temp\ipykernel_460\991912815.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contagem_modelos = df.groupby('nome_marca')['nome_modelo'].nunique().reset_index()


,Marca,Porcentagem (%),QTD Modelos
0,MERCEDES-BENZ,9.34,853
1,FORD,7.65,651
2,VW - VOLKSWAGEN,6.08,546
3,GM - CHEVROLET,5.60,551
4,FIAT,5.41,587
5,VOLVO,4.17,434
6,SCANIA,3.65,433
7,BMW,3.54,455
8,HONDA,3.39,304
9,RENAULT,2.90,337


## Tratamento para nomes de marcas repetidos

In [25]:
# pip install rapidfuzz
from rapidfuzz import fuzz, process

marcas = sorted(df['nome_marca'].cat.categories.tolist())

# Para cada marca, busca similares com score > 70
suspeitos = []
for marca in marcas:
    similares = process.extract(marca, marcas, scorer=fuzz.token_sort_ratio, limit=3)
    for similar, score, _ in similares:
        if similar != marca and score > 70:
            suspeitos.append((marca, similar, score))

# Remove duplicatas e exibe
vistos = set()
for a, b, score in sorted(suspeitos, key=lambda x: -x[2]):
    par = tuple(sorted([a, b]))
    if par not in vistos:
        vistos.add(par)
        print(f"{score:3}%  |  '{a}'  →  '{b}'")

91.66666666666666%  |  'REGAL PARTOR'  →  'REGAL RARTOR'
91.66666666666666%  |  'REGAL RAPTOR'  →  'REGAL RARTOR'
88.88888888888889%  |  'BUELL'  →  'BULL'
85.71428571428572%  |  'ASIA MOTORS'  →  'KIA MOTORS'
83.33333333333334%  |  'CHANA'  →  'CHANGAN'
83.33333333333334%  |  'REGAL PARTOR'  →  'REGAL RAPTOR'
80.0%  |  'CAB MOTORS'  →  'KIA MOTORS'
80.0%  |  'KIMCO'  →  'KYMCO'
80.0%  |  'VOLKSWAGEN'  →  'VW - VOLKSWAGEN'
78.26086956521739%  |  'CHEVROLET'  →  'GM - CHEVROLET'
77.77777777777779%  |  'ASIA MOTORS'  →  'SIAMOTO'
77.77777777777779%  |  'MOTOMORINI'  →  'MOTORINO'
76.92307692307692%  |  'AGRALE'  →  'LAVRALE'
76.92307692307692%  |  'BEPOBUS'  →  'NEOBUS'
76.92307692307692%  |  'BRAVA'  →  'FIBRAVAN'
76.92307692307692%  |  'CAOA CHERY'  →  'CAOA CHERY/CHERY'
76.92307692307692%  |  'DAYANG'  →  'SANYANG'
76.19047619047619%  |  'CAB MOTORS'  →  'ASIA MOTORS'
75.0%  |  'BETA'  →  'NETA'
75.0%  |  'DAF'  →  'DAFRA'
75.0%  |  'SANYANG'  →  'SSANGYONG'
75.0%  |  'WAKE'  →  'WALK

In [26]:
MAPA_MARCAS = {
    # Claramente a mesma marca
    'VW - VOLKSWAGEN' : 'VOLKSWAGEN',
    'GM - CHEVROLET'  : 'CHEVROLET',
    'CAOA CHERY/CHERY': 'CAOA CHERY',
    
    # Typos no dataset
    'REGAL PARTOR'    : 'REGAL RAPTOR',
    'REGAL RARTOR'    : 'REGAL RAPTOR',
    'KIMCO'           : 'KYMCO',
}

df['nome_marca'] = df['nome_marca'].replace(MAPA_MARCAS)

C:\Users\nicol\AppData\Local\Temp\ipykernel_460\3976272845.py:13: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['nome_marca'] = df['nome_marca'].replace(MAPA_MARCAS)


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5018080 entries, 0 to 5229871
Data columns (total 8 columns):
 #   Column            Dtype   
---  ------            -----   
 0   nome_marca        category
 1   nome_modelo       category
 2   tipo_veiculo      category
 3   nome_combustivel  category
 4   ano_referencia    int16   
 5   ano_modelo        float64 
 6   valor_reais       float64 
 7   idade_veiculo     float64 
dtypes: category(4), float64(3), int16(1)
memory usage: 191.8 MB


In [28]:
def segmentar_preco(valor):
    if valor <= 50000 : return 'Entrada'
    if valor <= 150000 : return 'Intermediário'
    if valor <= 350000 : return 'Premium'
    return 'Luxo'

df['segmentacao_veiculo'] = df['valor_reais'].apply(segmentar_preco)
df.head()

,nome_marca,nome_modelo,tipo_veiculo,nome_combustivel,ano_referencia,ano_modelo,valor_reais,idade_veiculo,segmentacao_veiculo
0,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3231.0,26.0,Entrada
1,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3181.0,26.0,Entrada
2,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3189.0,26.0,Entrada
3,ADLY,Atv 100,moto,Gasolina,2026,2000.0,3197.0,26.0,Entrada
4,ADLY,Atv 100,moto,Gasolina,2025,2000.0,3204.0,25.0,Entrada


In [29]:
df.to_parquet(PATH_SILVER, index = False, compression = 'zstd')
print('Salvo na Silver com sucesso')

Salvo na Silver com sucesso
